# 02 - Jumper et al. 2021: AlphaFold2 Reproduction

This notebook is a **from-scratch PyTorch reproduction** of AlphaFold2's main ideas. It does not use the official AlphaFold repository, runners, model parameters, or APIs.

We reproduce the reasoning rather than the exact proprietary training recipe:

1. MSA and pair representations are processed jointly.
2. Evoformer-like blocks exchange information between MSA rows and residue pairs.
3. A structure module predicts coordinates and confidence heads.
4. Recycling feeds predictions back through the network.
5. CASP14-style scoring compares predicted structures to held-out targets.

## Layout and cluster assumptions

Your hardware report says an A100 80GB is available. That is enough for realistic inference and medium-size training experiments, but full AF2-scale training remains extremely expensive. The notebook is written to scale: small synthetic smoke batches now, real feature tensors and distributed training later.

In [ ]:
from __future__ import annotations

import json
import math
import os
import random
import shlex
import subprocess
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "data"
MODEL_ROOT = PROJECT_ROOT / "models"
RESULTS_ROOT = PROJECT_ROOT / "results"
RUNS_ROOT = PROJECT_ROOT / "runs"

for path in [DATA_ROOT, MODEL_ROOT, RESULTS_ROOT, RUNS_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

def latest_environment_report() -> dict:
    report_dir = DATA_ROOT / "environment_reports"
    reports = sorted(report_dir.glob("environment_report_*_utc.json"))
    if not reports:
        return {}
    return json.loads(reports[-1].read_text(encoding="utf-8"))

ENV_REPORT = latest_environment_report()

def seed_everything(seed: int = 7) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def device() -> torch.device:
    return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

def write_text(path: Path, text: str) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")
    return path

def run(cmd: list[str], *, cwd: Path | None = None, timeout: int = 30, dry_run: bool = True):
    print("$", " ".join(shlex.quote(str(x)) for x in cmd))
    if dry_run:
        print("DRY_RUN=True, command was not executed.")
        return None
    completed = subprocess.run(cmd, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=timeout, check=False)
    print(completed.stdout)
    if completed.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {completed.returncode}")
    return completed

def gpu_summary() -> str:
    devices = ENV_REPORT.get("torch", {}).get("devices", [])
    if devices:
        first = devices[0]
        return f"{first.get('name')} / {first.get('total_memory_gb')} GB / cc {first.get('major')}.{first.get('minor')}"
    return "No saved GPU report found."

seed_everything(7)
print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {device()}")
print(f"Saved cluster GPU: {gpu_summary()}")

In [ ]:
PAPER = "jumper_2021"
DATA_DIR = DATA_ROOT / PAPER
MODEL_DIR = MODEL_ROOT / PAPER
RESULT_DIR = RESULTS_ROOT / PAPER
RUN_DIR = RUNS_ROOT / PAPER

paths = {
    "raw": DATA_DIR / "raw",
    "features": DATA_DIR / "features",
    "labels": DATA_DIR / "labels",
    "checkpoints": MODEL_DIR / "checkpoints",
    "predictions": RESULT_DIR / "predictions",
    "metrics": RESULT_DIR / "metrics",
    "slurm": RUN_DIR / "slurm",
}
for p in paths.values():
    p.mkdir(parents=True, exist_ok=True)
print(json.dumps({k: str(v) for k, v in paths.items()}, indent=2))

## Step 1 - Benchmark contract

For faithful reproduction, train using data available before CASP14 and use the CASP14 template boundary. Enhanced runs can use newer databases or architectures, but must be separate experiment rows.

Scientifically, AF2's CASP14 result was a generalization claim under a temporal information boundary. If we train or template against later structures, we are no longer asking whether the architecture reproduces the paper; we are asking a different, easier question.

Computationally, the contract becomes metadata attached to every feature tensor and checkpoint. Mathematically, it fixes the train/test split and the allowed conditioning variables, so score changes can be attributed to model or input changes instead of hidden leakage.

In [ ]:
contract = {
    "paper": "Jumper et al. 2021",
    "benchmark": "CASP14 monomer targets",
    "max_template_date": "2020-05-14",
    "implementation": "own_pytorch",
    "uses_official_weights_or_api": False,
}
write_text(DATA_DIR / "benchmark_contract.json", json.dumps(contract, indent=2))
print(json.dumps(contract, indent=2))

## Step 2 - Feature representation

AF2's practical breakthrough is not just a larger network; it is the representation. The MSA track stores evolutionary variation across sequences, while the pair track stores residue-residue geometry. The two tracks repeatedly talk to each other.

Scientifically, the MSA dimension captures evolutionary experiments performed by nature, and the pair dimension captures geometric relationships that must become consistent in 3D. AF2's strength comes from letting these two views iteratively constrain each other.

Computationally, we keep two tensors: `msa` with shape `[N_msa, L, C_msa]` and `pair` with shape `[L, L, C_pair]`. Mathematically, row/column operations on the MSA estimate residue-specific and co-evolutionary signals, while pair updates approximate a learned function over residue-residue potentials.

In [ ]:
class AF2FeatureDataset(Dataset):
    def __init__(self, feature_dir: Path, synthetic_if_empty: bool = True, n_synthetic: int = 6, n_msa: int = 32, length: int = 96):
        self.files = sorted(feature_dir.glob("*.npz"))
        self.synthetic_if_empty = synthetic_if_empty
        self.n_synthetic = n_synthetic
        self.n_msa = n_msa
        self.length = length

    def __len__(self):
        return len(self.files) if self.files else (self.n_synthetic if self.synthetic_if_empty else 0)

    def __getitem__(self, idx):
        if self.files:
            arr = np.load(self.files[idx])
            return {k: torch.as_tensor(arr[k]) for k in arr.files}
        N, L = self.n_msa, self.length
        msa = F.one_hot(torch.randint(0, 22, (N, L)), 22).float()
        residue_index = torch.arange(L)
        relpos = (residue_index[:, None] - residue_index[None, :]).clamp(-32, 32) + 32
        pair = F.one_hot(relpos, 65).float()
        true_ca = torch.cumsum(torch.randn(L, 3) * 0.5 + torch.tensor([3.8, 0.0, 0.0]), dim=0)
        return {"msa": msa, "pair": pair, "true_ca": true_ca}

dataset = AF2FeatureDataset(paths["features"])
sample = dataset[0]
print({k: tuple(v.shape) for k, v in sample.items()})

## Step 3 - Evoformer-like blocks

This is a compact research implementation, not a line-by-line clone. It keeps the essential message passing:

- MSA row attention: compare residues within each aligned sequence.
- MSA column summary: aggregate evolutionary signal into the pair track.
- Pair update: reason over residue-pair features.

Full AF2 adds triangle multiplication/attention, extra normalization details, template embedding, recycling features, and many loss heads. We add those incrementally after the baseline trains.

Scientifically, an Evoformer block is a consistency engine: residue-pair beliefs should agree with MSA evidence and with other pairs that form triangles in 3D space. A contact between `i` and `k` plus a contact between `k` and `j` constrains what can be true about `i` and `j`.

Computationally, this compact version uses attention and feed-forward updates rather than the full AF2 block, keeping the notebook runnable while preserving the information flow. Mathematically, the block alternates learned transformations `MSA <- f(MSA, pair)` and `pair <- g(pair, MSA summary)`, approximating iterative message passing over a complete residue graph.

In [ ]:
class EvoformerLiteBlock(nn.Module):
    def __init__(self, msa_dim: int, pair_dim: int, heads: int = 4):
        super().__init__()
        self.msa_attn = nn.MultiheadAttention(msa_dim, heads, batch_first=True)
        self.msa_ff = nn.Sequential(nn.LayerNorm(msa_dim), nn.Linear(msa_dim, 4 * msa_dim), nn.GELU(), nn.Linear(4 * msa_dim, msa_dim))
        self.outer = nn.Linear(msa_dim, pair_dim)
        self.pair_conv = nn.Sequential(
            nn.LayerNorm(pair_dim),
            nn.Linear(pair_dim, 4 * pair_dim),
            nn.GELU(),
            nn.Linear(4 * pair_dim, pair_dim),
        )

    def forward(self, msa, pair):
        B, N, L, C = msa.shape
        msa_flat = msa.reshape(B * N, L, C)
        attn, _ = self.msa_attn(msa_flat, msa_flat, msa_flat, need_weights=False)
        msa = (msa_flat + attn).reshape(B, N, L, C)
        msa = msa + self.msa_ff(msa)
        summary = msa.mean(dim=1)
        pair_update = self.outer(summary[:, :, None, :] + summary[:, None, :, :])
        pair = pair + pair_update
        pair = pair + self.pair_conv(pair)
        return msa, pair


class AF2Lite(nn.Module):
    def __init__(self, msa_in: int = 22, pair_in: int = 65, msa_dim: int = 128, pair_dim: int = 128, blocks: int = 6):
        super().__init__()
        self.msa_embed = nn.Linear(msa_in, msa_dim)
        self.pair_embed = nn.Linear(pair_in, pair_dim)
        self.blocks = nn.ModuleList([EvoformerLiteBlock(msa_dim, pair_dim) for _ in range(blocks)])
        self.coord_head = nn.Sequential(nn.LayerNorm(pair_dim + msa_dim), nn.Linear(pair_dim + msa_dim, 256), nn.GELU(), nn.Linear(256, 3))
        self.plddt_head = nn.Sequential(nn.LayerNorm(msa_dim), nn.Linear(msa_dim, 50))

    def forward(self, msa, pair, recycles: int = 1):
        msa = self.msa_embed(msa)
        pair = self.pair_embed(pair)
        for _ in range(recycles):
            for block in self.blocks:
                msa, pair = block(msa, pair)
        single = msa[:, 0]
        pair_context = pair.mean(dim=2)
        ca = self.coord_head(torch.cat([single, pair_context], dim=-1))
        plddt_logits = self.plddt_head(single)
        return {"ca": ca, "plddt_logits": plddt_logits, "pair": pair, "single": single}

model = AF2Lite().to(device())
with torch.no_grad():
    out = model(sample["msa"][None].to(device()), sample["pair"][None].to(device()))
print({k: tuple(v.shape) for k, v in out.items() if torch.is_tensor(v)})

## Step 4 - Structure losses

AF2 uses a rich loss suite. The minimal baseline here combines coordinate distance geometry and local confidence supervision. The coordinate loss is expressed through pairwise distances so it is rotation/translation invariant; later we can add frame-aligned point error, violation losses, distogram heads, and pLDDT calibration.

Scientifically, a protein prediction should be judged by geometry rather than absolute coordinate frame: rotating or translating a perfect model should not change the loss. Pairwise distances capture fold geometry without requiring a rigid alignment inside the training loop.

Computationally, `torch.cdist` gives a differentiable distance matrix, and smooth L1 reduces sensitivity to large early errors. Mathematically, the loss compares `D_hat_ij = ||x_hat_i - x_hat_j||` to `D_ij = ||x_i - x_j||`, plus a bond-length regularizer that biases neighboring C-alpha atoms toward plausible spacing.

In [ ]:
def pairwise_distance_loss(pred_ca, true_ca):
    pred_d = torch.cdist(pred_ca, pred_ca)
    true_d = torch.cdist(true_ca, true_ca)
    return F.smooth_l1_loss(pred_d, true_d)

def af2_lite_loss(outputs, true_ca):
    geom = pairwise_distance_loss(outputs["ca"], true_ca)
    bond = ((outputs["ca"][:, 1:] - outputs["ca"][:, :-1]).norm(dim=-1) - 3.8).pow(2).mean()
    return geom + 0.05 * bond

loader = DataLoader(dataset, batch_size=1, shuffle=True)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
model.train()
for step, item in enumerate(loader):
    item = {k: v.to(device()) for k, v in item.items()}
    opt.zero_grad(set_to_none=True)
    outputs = model(item["msa"], item["pair"], recycles=1)
    loss = af2_lite_loss(outputs, item["true_ca"])
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    print(f"step={step} loss={float(loss.detach().cpu()):.4f}")
    if step >= 2:
        break

## Step 5 - Recycling and inference

Recycling lets the model refine a structure hypothesis by feeding information from an earlier pass back into the network. The compact model above has a `recycles` argument; the next implementation iteration should inject previous distance/coordinate embeddings into the pair representation rather than merely repeating the blocks.

Scientifically, folding is a self-consistency problem: an initial hypothesis reveals clashes, domain arrangements, and long-range constraints that can guide a second pass. AF2's recycling mechanism exploits that by making prediction iterative rather than one-shot.

Computationally, recycling trades runtime and memory for refinement. Mathematically, it applies a recurrence `h_{t+1} = F(h_t, features, structure_t)` and hopes the sequence converges toward a geometrically consistent fixed point. In this first notebook, repeated blocks are a scaffold for the fuller recurrence.

In [ ]:
model.eval()
with torch.no_grad():
    item = {k: v[None].to(device()) for k, v in dataset[0].items()}
    one = model(item["msa"], item["pair"], recycles=1)["ca"]
    three = model(item["msa"], item["pair"], recycles=3)["ca"]
print({"one_recycle_shape": tuple(one.shape), "three_recycle_shape": tuple(three.shape)})

## Step 6 - Scoreboard and improvement track

The faithful row should use CASP14 targets, dated training/template boundaries, and this PyTorch implementation. Enhanced rows can use larger models, more recycles, modern databases, or new rerankers.

Scientifically, improvement claims need paired comparisons: the same targets, the same scoring metrics, and one controlled change at a time. Otherwise a better headline number may simply reflect a different benchmark surface.

Computationally, the registry is a small experiment database. Mathematically, each row defines a function from inputs to structures plus a metric vector; later we can compute deltas, confidence intervals, per-target wins/losses, and failure clusters.

In [ ]:
experiments = [
    {"name": "af2_lite_faithful_casp14", "faithful": True, "change": "own Evoformer-lite, CASP14 boundary"},
    {"name": "af2_lite_triangle_updates", "faithful": True, "change": "add triangle multiplication and attention"},
    {"name": "af2_lite_more_recycles", "faithful": False, "change": "extra inference recycles and seeds"},
    {"name": "af2_lite_modern_msa", "faithful": False, "change": "modern sequence databases"},
]
write_text(RUN_DIR / "experiment_registry.json", json.dumps(experiments, indent=2))
print(json.dumps(experiments, indent=2))

## Cluster execution template

The notebooks are deliberately importable and runnable from `papermill`, `jupyter nbconvert --execute`, or ordinary notebook execution. For long runs on the cluster, the code writes a small SLURM script that executes this notebook with parameters rather than keeping GPU time tied to the browser session.

In [ ]:
slurm = paths["slurm"] / "train_af2_lite.sbatch"
slurm.write_text(f"""#!/usr/bin/env bash
#SBATCH --job-name=af2-lite
#SBATCH --partition=gpu
#SBATCH --gres=gpu:1
#SBATCH --cpus-per-task=16
#SBATCH --mem=128G
#SBATCH --time=48:00:00
#SBATCH --output={paths['slurm']}/%x-%j.out
set -euo pipefail
cd "{PROJECT_ROOT}"
jupyter nbconvert --to notebook --execute 02_Jumper_2021_AlphaFold2_CASP14_Reproduction.ipynb --output results/jumper_2021/executed.ipynb
""", encoding="utf-8")
print(slurm.read_text(encoding="utf-8"))